# Revision-Aware Repair Verification (RARV)
### Gemma 4 Developer Agent Paper Track — readable CPU reviewer companion (v1.8)

**Author:** Clifford Imaguezegie  
**Purpose:** Read the research architecture and reproduce the controller replay and archived-result reconciliation on Kaggle CPU.  
**Execution:** **One code cell** below. Click **Run All**, or run the cell titled **RUN — CPU replay and result reconciliation**. No preceding hidden setup cell is required.

This notebook contains **readable Python and Markdown only**: no embedded Base64/hex archives, executable encoded strings, or hidden evidence blocks. The original, immutable E17 ZIP is downloaded from the public repository when Internet is available, or supplied as a Kaggle Input dataset when Internet is unavailable.

**Source repository:** https://github.com/clifford-imaguezegie/revision-aware-repair-verification  
**Evidence package:** [Gemma4_RARV_CPU_Review_Package_v1_0.zip](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/blob/main/evidence/Gemma4_RARV_CPU_Review_Package_v1_0.zip)


## Architecture — implemented framework

The Gemma 4 software-repair agent is **non-deterministic**: it chooses investigations, tool calls and source edits. RARV places a **deterministic evidence controller** around those actions without dictating their order.

| Layer | Implemented components | Role |
|:--|:--|:--|
| **1. Autonomous repair** | Gemma 4 agent; repository tools | Interpret task; inspect code; edit; request tests; submit candidate |
| **2. Trusted verification** | Tool supervisor; production fingerprint; monotonic revision ledger; trusted pytest adapter | Observe source changes and issue test receipts bound to source, check definition and environment |
| **3. Verification policy** | B final gate; C edit-triggered recheck; shared submission gate | B marks invalidated passes **STALE**; C rechecks within budget and reports a current outcome; both enforce current evidence at submission |
| **4. Independent assurance** | Production patch capture; protected evaluator; digest-based reconciliation | Preserve the exact production diff, grade in a separate evaluator and distinguish captured quality from verified delivery |

**Execution and evidence flow:**  
**Gemma tool action** → **source observer and revision ledger** → **trusted check and B/C controller** → **shared submission gate**.  
In parallel, **production patch capture** → **independent protected grading**; both paths join by **attempt ID, revision and patch digest**.

A check becoming **stale** means its previous result no longer applies to the current revision. It does **not** establish a regression. A **confirmed regression** requires a fresh qualified failure of the registered check.


## Key research contributions

**C1 — Integrated, revision-aware verification architecture.** An actual implemented controller links code revisions and content digests to trusted test receipts, policy decisions, exact production patch capture, independent grading and accepted delivery. The novelty is the auditable integration of known techniques, **not** an invented testing algorithm or a new state-machine formalism.

**C2 — Matched empirical study of recheck opportunity and activation.** GP03 compared policies B and C on nine preselected task pairs using matched budgets, the same final gate and the same agent configuration. Each policy passed **6/9** protected tasks (paired difference **0.0 percentage points**, 95% bootstrap interval **−22.2 to +22.2**). Across GP03–GP05, **24 autonomous runs contained zero qualifying natural pass-to-edit exposures**. In two deliberately triggered GP04 pairs, C reported failures in **1.150 s** and **1.047 s** before inference. The effect of proactive rechecking *after a naturally occurring trigger* remains unestimated.

**C3 — Empirical distinction among assurance boundaries.** A current receipt does not prove that the test covers the reported defect; a protected-passing candidate does not prove that the patch was submitted. The trace cases include public/protected test disagreements, an injected fault retained despite a protected pass, and protected-passing deadline captures without verified delivery.

**Reproducibility contribution (supporting evidence).** The E17 review package runs the **frozen production controller and trusted adapter** on a synthetic Git fixture and reconciles saved results against SHA-256-pinned archived JSON members. It does **not** rerun Gemma's GPU trajectories or the private protected tests.


## Evaluation coverage and reproducibility boundaries

| Cohort | Task pairs | Agent runs | Protected passes | Verified deliveries |
|:--|--:|--:|:--|:--|
| GP03 primary autonomous | 9 | 18 | B **6/9**, C **6/9** | B **5/9**, C **4/9** |
| GP04 autonomous follow-up | 2 | 4 | B **1/2**, C **1/2** | Neither arm delivered |
| GP05 autonomous follow-up | 1 | 2 | B **0/1**, C **0/1** | Neither arm delivered |
| GP04 deliberate-trigger controlled | 2 | 4 | Captures passed protected suites in both arms | Neither arm delivered |

These cohorts have different starting conditions and denominators. The **primary statistical comparison** is GP03 only. The controlled results demonstrate mechanism activation and should not be pooled into a causal repair-success estimate.


## Reproduce the CPU checks on Kaggle

1. Open this notebook and select **CPU**. Use a supported Python environment (**Python 3.10 or newer**; this path has been independently exercised on Python 3.13). Git must be available.
2. Ensure `Gemma4_RARV_CPU_Review_Package_v1_0.zip` is accessible. The code automatically looks under `/kaggle/input` and `/kaggle/working` for an attached copy, then tries the **public GitHub raw file** if Internet is enabled. The package must match SHA-256 `3b8c15a8433909c09fab11cce28f136c8ef9d195e2769f237c24b71315cd96c0`. No other inputs are silently substituted.
3. Run the **single code cell** below. It installs the bundled pinned **pure-Python** pytest wheels into a temporary isolated virtual environment, runs the frozen controller on the synthetic checkout, and reconciles the selected archived JSON evidence.
4. Review the **formatted tables** and the output folder reported by the code. The saved replay, reconciliation and detailed execution log remain available in Kaggle Outputs.

**Expected outputs:** controller replay `PASS`; six trusted tests; B `STALE` after a production edit; C `CURRENT_FAIL`; both broken gates blocked and restored gates accepted; GP03 protected **6/9** per policy and verified delivery **5/9 B**, **4/9 C**; 24 autonomous runs and zero qualifying natural triggers.

**Environment caveat:** E17 itself is frozen to Python 3.12. If Kaggle provides a different compatible version, the launcher copies the **single Python-version assertion** into a temporary compatibility script and records the different environment fingerprint; it does not rewrite the original source archive. This is a compatibility replay, not a byte-identical historical environment.

**Scope:** the bundled checker validates **12 selected original JSON members**. Full patch-level verification additionally requires **seven original sealed archives**, which are not embedded in this notebook. Fresh protected grading requires the organizer's evaluator and protected assets.

For human-readable source, see the [implementation notebooks](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/tree/main/notebooks), [evidence index](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/blob/main/docs/evidence-index.md), and [reproduction documentation](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/blob/main/evidence/REPRODUCE.md).


In [1]:
# RUN THIS CELL — CPU replay and archived-result reconciliation (NO prior setup cell)
# Everything here is clear-text Python. The E17 ZIP is a separate, checksum-pinned artifact.
from pathlib import Path
from zipfile import ZipFile
from IPython.display import display, Markdown
import hashlib, json, os, shutil, subprocess, sys, tempfile, urllib.request, venv

E17_NAME = 'Gemma4_RARV_CPU_Review_Package_v1_0.zip'
E17_SHA256 = '3b8c15a8433909c09fab11cce28f136c8ef9d195e2769f237c24b71315cd96c0'
E17_GITHUB_RAW = ('https://raw.githubusercontent.com/clifford-imaguezegie/'
                  'revision-aware-repair-verification/main/evidence/' + E17_NAME)
# Optional: set to an exact attached path. Leave None for automatic discovery.
E17_PACKAGE_PATH = None

if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10+ is needed for the pinned pytest 8.4.2 wheels.')
if not shutil.which('git'):
    raise RuntimeError('Git is required for the synthetic source-revision fixture.')
work = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
run_dir = Path(tempfile.mkdtemp(prefix='rarv_readable_', dir=work))

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def find_e17():
    if E17_PACKAGE_PATH:
        path = Path(E17_PACKAGE_PATH)
        if not path.is_file():
            raise FileNotFoundError(f'E17_PACKAGE_PATH does not exist: {path}')
        return path
    for base in [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd(), Path('/mnt/data')]:
        if base.is_dir():
            matches = sorted(base.rglob(E17_NAME))
            if matches:
                return matches[0]
    dest = run_dir / E17_NAME
    try:
        print('E17 ZIP was not attached. Attempting public GitHub download...')
        with urllib.request.urlopen(E17_GITHUB_RAW, timeout=25) as response:
            dest.write_bytes(response.read())
        return dest
    except Exception as ex:
        raise FileNotFoundError(
            f'Cannot find {E17_NAME}. In Kaggle, attach it as an Input dataset '
            f'(or enable Internet for the public GitHub download) and rerun this single cell. '
            f'Underlying download error: {type(ex).__name__}: {ex}'
        ) from ex

zip_path = find_e17()
if sha256(zip_path) != E17_SHA256:
    raise ValueError(f'E17 checksum mismatch for {zip_path}. Refusing unpinned evidence.')

with ZipFile(zip_path) as zf:
    corrupted = zf.testzip()
    if corrupted:
        raise ValueError(f'Archive CRC failure: {corrupted}')
    for member in zf.namelist():
        part = Path(member)
        if part.is_absolute() or '..' in part.parts:
            raise ValueError(f'Unsafe archive entry: {member}')
    zf.extractall(run_dir)
package = run_dir / Path(E17_NAME).stem

# An isolated environment uses the exact bundled wheel hashes, without network access.
venv_dir = run_dir / 'review_env'
venv.EnvBuilder(with_pip=False).create(venv_dir)
python = venv_dir / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
site_packages = subprocess.check_output(
    [str(python), '-I', '-c', "import sysconfig; print(sysconfig.get_path('purelib'))"], text=True
).strip()
install = subprocess.run(
    [sys.executable, '-m', 'pip', '--isolated', 'install', '--no-index',
     '--find-links', str(package / 'wheels'), '--require-hashes', '--ignore-installed',
     '--no-compile', '--target', site_packages, '-r', str(package / 'requirements.txt')],
    capture_output=True, text=True, timeout=150
)
if install.returncode:
    raise RuntimeError('Pinned offline install failed:\n' + install.stderr[-3500:])

# The original replay implementation remains untouched. Only its version ASSERTION
# may be relaxed in a separate, disposable compatibility runner.
compatibility = sys.version_info[:2] != (3, 12)
runner = 'replay_controller.py'
if compatibility:
    src = (package / runner).read_text()
    old = '''require(sys.version_info[:2] == (3, 12) and environment["pytest_version"] == "8.4.2",
            "Use Python 3.12 and the pinned offline pytest environment")'''
    new = '''require(environment["pytest_version"] == "8.4.2",
            "Use the pinned offline pytest environment")'''
    if src.count(old) != 1:
        raise RuntimeError('Pinned runner assertion does not match the expected freeze; no edit applied.')
    runner = 'replay_controller_kaggle_compat.py'
    (package / runner).write_text(src.replace(old, new, 1))

execution_env = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTEST_ADDOPTS')}
execution_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
log = []
for label, argv in [
    ('Actual controller replay', [runner, '--output', str(run_dir / 'replay')]),
    ('Archived results reconciliation', ['reconcile_results.py', '--output', str(run_dir / 'results.json')]),
]:
    print('Executing:', label)
    result = subprocess.run([str(python)] + argv, cwd=package, env=execution_env,
                            capture_output=True, text=True, timeout=180)
    log.append(f'{label}\n{result.stdout}\n{result.stderr}\nExit code {result.returncode}')
    if result.returncode:
        (run_dir / 'Execution_Log.txt').write_text('\n\n'.join(log))
        raise RuntimeError(f'{label} FAILED. See {run_dir / "Execution_Log.txt"}.\n' +
                           (result.stderr or result.stdout)[-3000:])
(run_dir / 'Execution_Log.txt').write_text('\n\n'.join(log))

replay = json.loads((run_dir / 'replay' / 'Replay_Report.json').read_text())
results = json.loads((run_dir / 'results.json').read_text())
assert replay['status'] == results['status'] == 'PASS'
assert replay['model_calls'] == 0 and sum(a['trusted_tests'] for a in replay['arms']) == 6
assert results['autonomous_runs'] == 24 and results['controlled_runs'] == 4
assert results['qualifying_autonomous_pass_to_edit_events'] == 0
assert results['verified_source_members'] == 12
B = next(a for a in replay['arms'] if a['arm'] == 'B')
C = next(a for a in replay['arms'] if a['arm'] == 'C')
primary = results['cohorts']['GP03_primary']
interval = results['GP03_primary_inference']['paired_repo_stratified_bootstrap_95']
mode = ('Pinned Python 3.12' if not compatibility else
        'Compatibility replay on Python ' + sys.version.split()[0])

# Markdown tables are intentionally simple to render cleanly in Kaggle.
summary = f'''# CPU reviewer checks — PASS

**Execution:** {mode} | **pytest:** {replay['pytest']} | **model calls:** 0 | **E17 SHA-256:** verified

### 1. Real controller and trusted-test replay

| Checkpoint | B: Final gate | C: Proactive recheck |
|:--|:--|:--|
| Scratch-only edit | {B['after_scratch_edit']} | {C['after_scratch_edit']} |
| Production edit | **{B['after_production_edit']}** | **{C['after_production_edit']}** |
| Broken source at gate | {B['broken_gate']} | {C['broken_gate']} |
| Restored source at gate | {B['restored_gate']} | {C['restored_gate']} |
| Trusted pytest executions | **{B['trusted_tests']}** | **{C['trusted_tests']}** |

### 2. Archive-derived recorded result reconstruction

| Metric | Policy B | Policy C |
|:--|--:|--:|
| GP03 protected passes | **{primary['protected_passes']['B']}/9** | **{primary['protected_passes']['C']}/9** |
| GP03 verified deliveries | **{primary['verified_deliveries']['B']}/9** | **{primary['verified_deliveries']['C']}/9** |

**C − B protected difference:** 0.0 percentage points  
**95% paired bootstrap interval:** {interval[0]*100:+.1f} to {interval[1]*100:+.1f} points  
**Autonomous runs:** {results['autonomous_runs']}; **qualifying pass-to-edit events:** 0  
**Controlled C detection:** Rich {results['controlled_detections']['rich_3718']:.3f}s; Requests {results['controlled_detections']['requests_7205']:.3f}s.

### 3. What has and has not been reproduced

- **Verified now:** original E17 package checksum, frozen controller on a synthetic Git fixture, six actual trusted pytest executions, source revision transitions, gate decisions and reconciliation of 12 archive JSON evidence members.
- **Not rerun:** Gemma 4 GPU inference trajectories, the organizer's private protected grading, or all 28 original patch-byte checks (which require seven sealed original archives).
- **Compatibility note:** a non-3.12 environment uses a separate runner with only the explicit Python-version assertion relaxed, so the environment fingerprint differs from the originally pinned study.

**Outputs:** `{run_dir}` (`replay/`, `results.json`, `Execution_Log.txt`, `README_RESULTS.md`).
'''
(run_dir / 'README_RESULTS.md').write_text(summary)
display(Markdown(summary))


Executing: Actual controller replay


Executing: Archived results reconciliation


# CPU reviewer checks — PASS

**Execution:** Compatibility replay on Python 3.13.5 | **pytest:** 8.4.2 | **model calls:** 0 | **E17 SHA-256:** verified

### 1. Real controller and trusted-test replay

| Checkpoint | B: Final gate | C: Proactive recheck |
|:--|:--|:--|
| Scratch-only edit | CURRENT_PASS | CURRENT_PASS |
| Production edit | **STALE** | **CURRENT_FAIL** |
| Broken source at gate | BLOCKED_VISIBLE_CHECK_FAILED | BLOCKED_VISIBLE_CHECK_FAILED |
| Restored source at gate | SUBMITTED_GATE_VALID | SUBMITTED_GATE_VALID |
| Trusted pytest executions | **3** | **3** |

### 2. Archive-derived recorded result reconstruction

| Metric | Policy B | Policy C |
|:--|--:|--:|
| GP03 protected passes | **6/9** | **6/9** |
| GP03 verified deliveries | **5/9** | **4/9** |

**C − B protected difference:** 0.0 percentage points  
**95% paired bootstrap interval:** -22.2 to +22.2 points  
**Autonomous runs:** 24; **qualifying pass-to-edit events:** 0  
**Controlled C detection:** Rich 1.150s; Requests 1.047s.

### 3. What has and has not been reproduced

- **Verified now:** original E17 package checksum, frozen controller on a synthetic Git fixture, six actual trusted pytest executions, source revision transitions, gate decisions and reconciliation of 12 archive JSON evidence members.
- **Not rerun:** Gemma 4 GPU inference trajectories, the organizer's private protected grading, or all 28 original patch-byte checks (which require seven sealed original archives).
- **Compatibility note:** a non-3.12 environment uses a separate runner with only the explicit Python-version assertion relaxed, so the environment fingerprint differs from the originally pinned study.

**Outputs:** `/mnt/data/rarv_readable_wvmit9z2` (`replay/`, `results.json`, `Execution_Log.txt`, `README_RESULTS.md`).


## Implementation notebooks (human-readable source)

The repository contains **17 implementation notebooks**, of which **16 have executable cells** and **GP04 Follow-up Evidence** is documentation-only. They preserve the original methods and code, but several require the organizer's Kaggle mounts, authorised archives, GPU hardware and frozen dependencies. Running a source notebook from scratch may reproduce the **procedure** without reproducing an identical stochastic GPU trajectory.

| Stage | What the notebooks reproduce | Requirements |
|:--|:--|:--|
| **GP00–GP01** | Task inventory, harness audit, baseline/reference qualification | Published tasks and snapshots, CPU + offline wheels |
| **GP02B–GP02C** | Development cases, ledger, adapter and worker attachment | CPU for controllers; live Gemma and L4s for autonomous attempts |
| **GP03** | CPU qualification, frozen B/C evaluation, protected grading, recovery | CPU for qualification; original model and grader inputs for GPU/independent grading |
| **GP04** | Follow-up evidence and experimental record | Document-only source; inspect sealed records |
| **GP05** | Supplemental paired experiment and grader | Original Kaggle runtime, model and archived records |

Every released implementation notebook includes its own upfront execution guide. All source code is visible as ordinary notebook code or readable Python, not as encoded embedded ZIP payloads. See the [implementation notebooks on GitHub](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/tree/main/notebooks) and [evidence index](https://github.com/clifford-imaguezegie/revision-aware-repair-verification/blob/main/docs/evidence-index.md).


## How to interpret the result

The CPU check confirms **mechanism and evidence-accounting reproducibility**, not fresh autonomous software repair. In the original controlled live study, C detected deliberately injected failures before inference, but no controlled arm achieved verified delivery; natural triggering was not observed in the 24 autonomous runs. These limitations are essential to interpreting the contribution.
